# Ders 09 - Üstbiliş Tasarım Deseni


## Kurulum

Bu not defteri, Microsoft Agent Framework kullanarak Metakognisyon tasarım desenini gösterir.

**Ön Koşullar:**
- Ortam değişkenleri aracılığıyla yapılandırılmış Azure OpenAI dağıtımı
- Azure CLI kimlik doğrulaması yapılmış (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Metabiliş Nedir?

Metabiliş, **düşünme üzerine düşünme** demektir. Yapay zeka ajanları bağlamında, şu özelliklere sahip ajanlar inşa etmek anlamına gelir:

- Kendi çıktıları ve akıl yürütme süreçleri üzerine **öz-yansıtma** yapmak
- **Hataları tespit etmek** ve sessizce başarısız olmak yerine sorunsuz toparlanmak
- Yanıtlarının eksiksiz ve faydalı olup olmadığını **değerlendirmek**
- İlk yaklaşım işe yaramadığında stratejilerini **uyarlamak** (örneğin, yedek sisteme geçmek)

Bir metabiliş ajanı sadece soruları yanıtlamakla kalmaz — kendi performansını izler ve anında ayar yapar.


## Birincil ve Yedek Araçlar

Yaygın bir üstbiliş deseni **yedekleme stratejisidir**. Ajan önce birincil aracı dener; eğer başarısız olursa (örneğin, 404 hatası), ajan hatayı fark eder ve şeffaf bir şekilde yedek araca geçer.

Bu, birincil hizmetlerin kullanılamadığı ve ajanların alternatif bir yol seçmeden önce sorunu kendi kendine teşhis etmesi gereken gerçek dünya sistemlerini yansıtır.

Aşağıda iki uçuş arama aracını tanımlıyoruz:
- **Birincil** — Paris, Tokyo ve Barcelona'yı kapsar
- **Yedek** — Berlin, Sidney ve New York Şehri'ni kapsar


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Hata Kurtarmalı Kendini Değerlendiren Ajan

Aşağıdaki ajan, önce birincil uçuş sistemini denemesi, arızaları tanıması ve şeffaf bir şekilde yedek sisteme geçmesi talimatını almıştır. Her yanıtın ardından, kullanıcının sorusunu tam olarak yanıtlayıp yanıtlamadığı konusunda kısaca kendini değerlendirir.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Öz Değerlendirme Kalıbı

Metabilişin bir diğer yönü **öz değerlendirmedir**: ayrı bir temsilci (veya aynı temsilci ikinci bir geçişte) yanıtı bütünlük, doğruluk ve fayda açısından gözden geçirir.

Aşağıda seyahat acentesi yanıtlarını üç boyutta puanlayan bir `ResponseEvaluator` temsilcisi oluşturuyoruz.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Özet

Bu derste Microsoft Agent Framework kullanarak **metabilişsel ajanlar** oluşturmayı öğrendiniz:

- **Öz-yansıtma**: Kendi muhakemelerini izleyen ve neler olduğunu şeffaf bir şekilde ileten ajanlar.
- **Yedeklerle hata kurtarma**: Ajanın hataları (örneğin 404 hataları) tespit edip otomatik olarak alternatif bir kaynağı deneyen birincil + yedek araç deseni.
- **Öz-değerlendirme**: Yanıtları tamlık, doğruluk ve yardımcı olma açısından puanlayan ayrı bir değerlendirme ajanı.

Bu desenler ajanları daha sağlam, şeffaf ve güvenilir yapar — bu, üretim ortamları için kritik niteliklerdir.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
